# 🧠 Track 1: Pure LLM Fine-Tuning Pipeline (QLoRA + SFT)
## Domain Adaptation for Corporate Litigation Knowledge Graphs & Ontologies

### 🎯 Research Objective
In Phase 1, our zero-training baseline established that untuned `Qwen2.5-1.5B-Instruct` suffers from severe bottlenecks on structured graph tasks:
- **Task A (Triplet Extraction)**: **25.07% Macro F1** (fails to conform to ontology vocabulary, inverts relation directions).
- **Task B (Multi-Hop Deduction)**: **92% (2-Hop)** and **86% (3-Hop)**.
- **Task C (Link Prediction)**: **24.00% Accuracy** (near random guess level).

**In Track 1**, we test the **Pure LLM Paradigm**: can textual QLoRA fine-tuning ($r=16, \alpha=32$) on 1,200 domain-supervised samples close this performance gap on unseen corporate litigation scenarios?

---
### 🛠️ Hardware & Environment
- **Target Runtime**: Google Colab **Tesla T4 GPU (16GB VRAM)**.
- **Quantization**: 4-bit NormalFloat (NF4) base model (~1.1 GB VRAM).
- **Trainable Parameters**: ~18.5M (~1.2% of total weights).

### Step 1: Install Dependencies & Hardware Audit

In [ ]:
# Run in Google Colab:
!pip install -q transformers peft trl accelerate bitsandbytes datasets tqdm

In [ ]:
import sys
import os
import json
import time
import torch
import transformers

print("=" * 65)
print("🚀 HARDWARE & ACCELERATOR AUDIT")
print("=" * 65)
print(f"Python Version       : {sys.version.split()[0]}")
print(f"PyTorch Version      : {torch.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total           : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("GPU Device Name      : CPU Mode (Quantization disabled)")
print(f"Transformers Version : {transformers.__version__}")
print("=" * 65)

### Step 2: Clone/Sync Repository & Locate Processed Datasets

In [ ]:
from pathlib import Path

# 1. Clone or pull repository in Colab
if os.path.exists("/content") and not os.path.exists("/content/Fine-Tuning-PoCs"):
    !git clone https://github.com/sagarsambhwani/Fine-Tuning-PoCs.git
elif os.path.exists("/content/Fine-Tuning-PoCs"):
    !cd /content/Fine-Tuning-PoCs && git pull origin main

# 2. Locate repository directory
possible_roots = [
    "/content/Fine-Tuning-PoCs/llm-graph-ontology",
    "/content/Fine-Tuning-PoCs",
    "llm-graph-ontology",
    os.getcwd()
]
repo_root = None
for r in possible_roots:
    if os.path.exists(os.path.join(r, "src/data")) or os.path.exists(os.path.join(r, "data/processed")):
        repo_root = r
        break
if repo_root is None:
    repo_root = os.getcwd()

if repo_root not in sys.path:
    sys.path.append(repo_root)

# 3. Locate dataset files
data_dir = os.path.join(repo_root, "data/processed") if os.path.exists(os.path.join(repo_root, "data/processed")) else os.path.join(repo_root, "llm-graph-ontology/data/processed")
train_file = os.path.join(data_dir, "train.jsonl")
val_file = os.path.join(data_dir, "val.jsonl")
manifest_file = os.path.join(data_dir, "manifest.json")

# 4. Auto-generate if not yet generated on local disk (~2s)
if not os.path.exists(train_file) or not os.path.exists(manifest_file):
    print("⏳ Datasets not found on disk. Auto-generating 1,200 training & 150 val samples now...")
    gen_script = os.path.join(repo_root, "src/data/generate_track1_training_data.py")
    if not os.path.exists(gen_script):
        gen_script = os.path.join(repo_root, "llm-graph-ontology/src/data/generate_track1_training_data.py")
    if os.path.exists(gen_script):
        !python {gen_script}
    
    # Refresh paths
    for candidate in [os.path.join(repo_root, "data/processed"), os.path.join(repo_root, "llm-graph-ontology/data/processed"), "data/processed", "llm-graph-ontology/data/processed", "/content/Fine-Tuning-PoCs/llm-graph-ontology/data/processed"]:
        if os.path.exists(os.path.join(candidate, "train.jsonl")):
            data_dir = candidate
            break
    train_file = os.path.join(data_dir, "train.jsonl")
    val_file = os.path.join(data_dir, "val.jsonl")
    manifest_file = os.path.join(data_dir, "manifest.json")

with open(manifest_file, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"✅ Dataset verified: {manifest['dataset_name']}")
print(f"   Train Samples : {manifest['total_train_samples']} ({manifest['train_task_breakdown']})")
print(f"   Val Samples   : {manifest['total_val_samples']} ({manifest['val_task_breakdown']})")
print(f"   Anti-Leakage  : {manifest['anti_leakage_status']}")

### Step 3: Load Tokenizer & 4-Bit NF4 Base Model (`Qwen2.5-1.5B-Instruct`)
We enforce `torch.float32` precision on all trainable LoRA adapters and `model.config.torch_dtype = torch.float16` to eliminate Turing T4 `GradScaler` unscale conflicts.

In [19]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading base tokenizer & model: {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)
model.config.torch_dtype = torch.float16

model = prepare_model_for_kbit_training(model)

# LoRA Setup: r=16, alpha=32 across all 7 linear projection layers
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none"
)

model = get_peft_model(model, peft_config)
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)

trainable, total = model.get_nb_trainable_parameters()
print(f"✅ Model Loaded in 4-bit NF4 with LoRA Adapter injected!")
print(f"   Trainable Parameters: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

Loading base tokenizer & model: Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Model Loaded in 4-bit NF4 with LoRA Adapter injected!
   Trainable Parameters: 18,464,768 / 1,562,179,072 (1.18%)


### Step 4: Prepare Datasets & Configure SFTTrainer
**T4 Compatibility Note**: `fp16=False` and `bf16=False` in SFTConfig avoids `GradScaler` unscale exceptions on Turing GPUs, while `bnb_4bit_compute_dtype=torch.float16` and `paged_adamw_8bit` keep forward passes and optimizer memory low.

In [20]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

def prepare_dataset_from_jsonl(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f if line.strip()]
    formatted = []
    for r in records:
        text = tokenizer.apply_chat_template(r["messages"], tokenize=False)
        formatted.append({"text": text})
    return Dataset.from_list(formatted)

train_dataset = prepare_dataset_from_jsonl(train_file)
val_dataset = prepare_dataset_from_jsonl(val_file)

print(f"Loaded {len(train_dataset)} train samples and {len(val_dataset)} val samples.")

OUTPUT_DIR = "models/qwen-1.5b-legal-graph-adapter"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    weight_decay=0.01,
    logging_steps=15,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    fp16=False,                   # Disables GradScaler conflict on T4 GPU
    bf16=False,
    max_grad_norm=0.3,
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    dataset_text_field="text",
    max_length=512,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=sft_config,
    processing_class=tokenizer
)

print("✅ SFTTrainer configured with SFTConfig. Ready to train.")

Loaded 1200 train samples and 150 val samples.


Adding EOS to train dataset:   0%|          | 0/1200 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1200 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1200 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1200 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

✅ SFTTrainer configured with SFTConfig. Ready to train.


### Step 5: Execute Training (QLoRA SFT)

In [21]:
print("=" * 65)
print("🚀 STARTING QLORA SFT TRAINING (3 EPOCHS, EFF. BATCH 16)")
print("=" * 65)
start_time = time.time()
train_result = trainer.train()
duration = time.time() - start_time

print(f"\n🎉 Training Complete in {duration/60:.2f} minutes!")
print(f"   Final Train Loss: {train_result.training_loss:.4f}")

# Save trained adapter and tokenizer
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ LoRA adapter saved to: {OUTPUT_DIR}")

🚀 STARTING QLORA SFT TRAINING (3 EPOCHS, EFF. BATCH 16)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.132076,0.103239,0.106776,172162.000000,0.971802
100,0.085106,0.081395,0.090791,340181.000000,0.974002
150,0.078361,0.078612,0.084599,512142.000000,0.973809
200,0.074805,0.076633,0.079675,684416.000000,0.973993
225,0.075473,0.076584,0.079713,768213.000000,0.974055



🎉 Training Complete in 31.51 minutes!
   Final Train Loss: 0.2430
✅ LoRA adapter saved to: models/qwen-1.5b-legal-graph-adapter


### Step 6: Load Benchmark Test Suites & Run Post-Training Evaluation

In [22]:
# Switch model to eval mode
model.eval()

def load_test_suite(filename):
    candidates = [
        f"data/benchmark/{filename}",
        f"../data/benchmark/{filename}",
        f"llm-graph-ontology/data/benchmark/{filename}",
        f"/content/Fine-Tuning-PoCs/llm-graph-ontology/data/benchmark/{filename}"
    ]
    for c in candidates:
        if os.path.exists(c):
            with open(c, "r", encoding="utf-8") as f:
                return json.load(f)
    raise FileNotFoundError(f"Cannot locate {filename}")

test_task_a = load_test_suite("legal_task_a_triplets.json")
test_task_b = load_test_suite("legal_task_b_multihop.json")
test_task_c = load_test_suite("legal_task_c_link_pred.json")

print(f"Loaded Benchmark Suites: Task A ({len(test_task_a)}), Task B ({len(test_task_b)}), Task C ({len(test_task_c)})")

Loaded Benchmark Suites: Task A (50), Task B (50), Task C (50)


### Step 7: Run Full 50-Sample Evaluation on Fine-Tuned Model

In [23]:
from tqdm import tqdm
import re

def extract_triples_tuned(text_passage):
    prompt_messages = [
        {
            "role": "system",
            "content": (
                "You are a formal corporate litigation knowledge graph extractor. "
                "Extract factual triples strictly conforming to the legal ontology relations: "
                "[FILES_CLAIM_AGAINST, ADJUDICATED_BY, APPLIES_PRECEDENT, ESTABLISHES_REMEDY, LIABLE_FOR]. "
                "Output ONLY a valid JSON list of objects with keys: 'subject', 'predicate', 'object'."
            )
        },
        {
            "role": "user",
            "content": f"Extract legal triples from this text:\n{text_passage}"
        }
    ]
    formatted = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=150, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def query_multihop_tuned(context, question):
    prompt_messages = [
        {
            "role": "system",
            "content": "You are a legal knowledge graph deductive reasoning engine. Answer the question using ONLY the facts provided. State the target entity name directly."
        },
        {
            "role": "user",
            "content": f"{context}\n\nQuestion: {question}"
        }
    ]
    formatted = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=50, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def predict_link_tuned(head, tail, prompt_text):
    prompt_messages = [
        {
            "role": "system",
            "content": "Predict the single valid ontology relation connecting Head to Tail. Reply with ONLY the relation name."
        },
        {
            "role": "user",
            "content": prompt_text
        }
    ]
    formatted = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=25, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def parse_triples_json(raw_text):
    cleaned = re.sub(r'```(?:json)?', '', raw_text).strip('` \n')
    try:
        data = json.loads(cleaned)
        if isinstance(data, list):
            return set((item.get('subject', '').strip().lower(), item.get('predicate', '').strip().lower(), item.get('object', '').strip().lower()) for item in data if isinstance(item, dict))
    except Exception:
        pass
    return set()

# -----------------------------------------------------
# 1. Batch Evaluate Fine-Tuned Task A (Triplet IE)
# -----------------------------------------------------
print("Evaluating Fine-Tuned Model on Task A (50 samples)...")
tuned_prec_list, tuned_rec_list, tuned_f1_list = [], [], []

for item in tqdm(test_task_a, desc="Task A Evaluation"):
    pred_raw = extract_triples_tuned(item['input_text'])
    pred_triples = parse_triples_json(pred_raw)
    true_triples = set((t['subject'].strip().lower(), t['predicate'].strip().lower(), t['object'].strip().lower()) for t in item['target_triples'])
    
    tp = len(pred_triples.intersection(true_triples))
    prec = tp / len(pred_triples) if len(pred_triples) > 0 else 0.0
    rec = tp / len(true_triples) if len(true_triples) > 0 else 0.0
    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    tuned_prec_list.append(prec)
    tuned_rec_list.append(rec)
    tuned_f1_list.append(f1)

# -----------------------------------------------------
# 2. Batch Evaluate Fine-Tuned Task B (Multi-Hop)
# -----------------------------------------------------
print("\nEvaluating Fine-Tuned Model on Task B (50 samples)...")
tuned_hop2_correct = 0
tuned_hop3_correct = 0

for item in tqdm(test_task_b, desc="Task B Evaluation"):
    ans_2 = query_multihop_tuned(item['subgraph_context'], item['query_2hop']).strip().lower()
    ans_3 = query_multihop_tuned(item['subgraph_context'], item['query_3hop']).strip().lower()
    if item['target_2hop'].strip().lower() in ans_2:
        tuned_hop2_correct += 1
    if item['target_3hop'].strip().lower() in ans_3:
        tuned_hop3_correct += 1

# -----------------------------------------------------
# 3. Batch Evaluate Fine-Tuned Task C (Link Prediction)
# -----------------------------------------------------
print("\nEvaluating Fine-Tuned Model on Task C (50 samples)...")
tuned_link_correct = 0

for item in tqdm(test_task_c, desc="Task C Evaluation"):
    pred_rel = predict_link_tuned(item['head'], item['tail'], item['prompt']).strip().lower()
    if item['target_relation'].strip().lower() in pred_rel:
        tuned_link_correct += 1

# -----------------------------------------------------
# 📊 PRINT OFFICIAL COMPARISON TABLE: BASELINE VS TUNED
# -----------------------------------------------------
tuned_avg_prec = sum(tuned_prec_list) / len(tuned_prec_list)
tuned_avg_rec = sum(tuned_rec_list) / len(tuned_rec_list)
tuned_avg_f1 = sum(tuned_f1_list) / len(tuned_f1_list)
tuned_hop2_acc = tuned_hop2_correct / len(test_task_b)
tuned_hop3_acc = tuned_hop3_correct / len(test_task_b)
tuned_link_acc = tuned_link_correct / len(test_task_c)

# Baseline numbers from Phase 1
base_prec, base_rec, base_f1 = 0.3067, 0.2133, 0.2507
base_hop2, base_hop3 = 0.9200, 0.8600
base_link = 0.2400

print('\n' + '=' * 75)
print('📊 OFFICIAL HEAD-TO-HEAD: UNTUNED BASELINE VS. TRACK 1 FINE-TUNED')
print('=' * 75)
print(f'{"Metric / Task":<36} | {"Untuned Base":<14} | {"Track 1 Tuned":<14} | {"Delta (Δ)"}')
print('-' * 75)
print(f'{"Task A: Triplet Precision":<36} | {base_prec*100:>12.2f}% | {tuned_avg_prec*100:>12.2f}% | {(tuned_avg_prec - base_prec)*100:>+6.2f}%')
print(f'{"Task A: Triplet Recall":<36} | {base_rec*100:>12.2f}% | {tuned_avg_rec*100:>12.2f}% | {(tuned_avg_rec - base_rec)*100:>+6.2f}%')
print(f'{"Task A: Triplet Macro F1":<36} | {base_f1*100:>12.2f}% | {tuned_avg_f1*100:>12.2f}% | {(tuned_avg_f1 - base_f1)*100:>+6.2f}%')
print('-' * 75)
print(f'{"Task B: 2-Hop Deductive Acc.":<36} | {base_hop2*100:>12.2f}% | {tuned_hop2_acc*100:>12.2f}% | {(tuned_hop2_acc - base_hop2)*100:>+6.2f}%')
print(f'{"Task B: 3-Hop Deductive Acc.":<36} | {base_hop3*100:>12.2f}% | {tuned_hop3_acc*100:>12.2f}% | {(tuned_hop3_acc - base_hop3)*100:>+6.2f}%')
print('-' * 75)
print(f'{"Task C: Link Prediction Acc.":<36} | {base_link*100:>12.2f}% | {tuned_link_acc*100:>12.2f}% | {(tuned_link_acc - base_link)*100:>+6.2f}%')
print('=' * 75)

track1_results = {
    "domain": "CorporateLitigationOntology",
    "model": "Qwen/Qwen2.5-1.5B-Instruct + QLoRA SFT (Track 1)",
    "task_a_precision": round(tuned_avg_prec, 4),
    "task_a_recall": round(tuned_avg_rec, 4),
    "task_a_f1": round(tuned_avg_f1, 4),
    "task_b_hop2_accuracy": round(tuned_hop2_acc, 4),
    "task_b_hop3_accuracy": round(tuned_hop3_acc, 4),
    "task_c_link_accuracy": round(tuned_link_acc, 4)
}
with open("legal_track1_finetuned_results.json", "w", encoding="utf-8") as f:
    json.dump(track1_results, f, indent=2)
print("Saved Track 1 fine-tuned results to legal_track1_finetuned_results.json")

Evaluating Fine-Tuned Model on Task A (50 samples)...


Task A Evaluation: 100%|██████████| 50/50 [12:17<00:00, 14.74s/it]



Evaluating Fine-Tuned Model on Task B (50 samples)...


Task B Evaluation: 100%|██████████| 50/50 [01:30<00:00,  1.81s/it]



Evaluating Fine-Tuned Model on Task C (50 samples)...


Task C Evaluation: 100%|██████████| 50/50 [00:47<00:00,  1.05it/s]


📊 OFFICIAL HEAD-TO-HEAD: UNTUNED BASELINE VS. TRACK 1 FINE-TUNED
Metric / Task                        | Untuned Base   | Track 1 Tuned  | Delta (Δ)
---------------------------------------------------------------------------
Task A: Triplet Precision            |        30.67% |       100.00% | +69.33%
Task A: Triplet Recall               |        21.33% |       100.00% | +78.67%
Task A: Triplet Macro F1             |        25.07% |       100.00% | +74.93%
---------------------------------------------------------------------------
Task B: 2-Hop Deductive Acc.         |        92.00% |       100.00% |  +8.00%
Task B: 3-Hop Deductive Acc.         |        86.00% |        90.00% |  +4.00%
---------------------------------------------------------------------------
Task C: Link Prediction Acc.         |        24.00% |        36.00% | +12.00%
Saved Track 1 fine-tuned results to legal_track1_finetuned_results.json
